In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

## Import Libries

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
import os
import pandas as pd


## Config

In [26]:
ROOT = "/kaggle/input/competitions/industrial-surface-defect-detection/public"

#train
TRAIN_DIR = ROOT + '/train'
TEST_DIR = ROOT + '/test'
OUT = "/kaggle/working"

NUM_CLASSES = 4
CLASSES = ["defect_a", "defect_b", "defect_c", "defect_d"]
# IMG_SIZE 
HEIGHT = 128
WIDTH = 800


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cpu":
    print("WARNING: no GPU detected. On Kaggle:")


Using device: cpu


# Images Directories Inspection

In [4]:
train_files = os.listdir(TRAIN_DIR)
test_files = os.listdir(TEST_DIR)

print("Training entries:", len(train_files))
print("Test entries:", len(test_files))
print(" ")
# printing some files
print("First 10 training files:")
print(train_files[:10])
print(" ")
print("First 10 test files:")
print(test_files[:10])

Training entries: 11393
Test entries: 3000
 
First 10 training files:
['I2P3TOD3FY.jpg', '2GYBBED7DJ.jpg', 'DFRBLMRLQ9.jpg', 'I7GTCUX6HT.jpg', 'OMXK41V3QA.jpg', 'ZAIEQJE42A.jpg', 'F8EQ25BQXO.jpg', 'WH0JO3IVYP.jpg', 'A02HL9FILH.jpg', '7NJI6Y8I7H.jpg']
 
First 10 test files:
['LZ46LTL6O2.jpg', '13H1VTAZ42.jpg', 'G0DWU2GK1L.jpg', 'CVPHC0DT1P.jpg', '90XJFQDL7G.jpg', '3V8LZ56MOW.jpg', 'Y9EUV6TNP2.jpg', 'K76P42J0GK.jpg', 'ZSS7THB1UM.jpg', '8JNHXLSBU3.jpg']


- Above examples shows that extensions are **.jpg** but lets check for more or if files doent have extention if not will asign that **.jpg** extension.

In [5]:
# print("Training files without extension or other than .jpg:")

# for file in train_files:
#     extension = os.path.splitext(file)[1].lower()

#     if extension != ".jpg":
#         print("File:", file)
#         print("Extension:", extension if extension else "No extension")

# print("Test files without extension or other than .jpg:")

# for file in test_files:
#     extension = os.path.splitext(file)[1].lower()

#     if extension != ".jpg":
#         print("File:", file)
#         print("Extension:", extension if extension else "No extension")

## **Load the training CSV**

In [6]:
train_df = pd.read_csv(ROOT + "/train.csv")

train_df_copy = train_df.copy()

train_df_copy["has_defect"] = (
    train_df_copy["mask_rle"].notna().astype(int)
)

print(f"Values in 'has_defect': {train_df_copy['has_defect'].unique()}")
print()

defect_count = len(train_df_copy[train_df_copy["has_defect"] == 1])
print(f"Number of rows having a defect: {defect_count}")

print()
train_df["mask_rle"] = train_df["mask_rle"].fillna("")

print("Dataset shape:", train_df.shape)
print()
print("Dataset columns:")
print(train_df.columns)

defect_class_map = {"no_defect": 0, "defect_a":1, "defect_b":2, "defect_c" : 3, "defect_d":4}

train_df_copy['defect_class'] = np.where(
    train_df_copy['mask_rle'].isna(),
    0,
    train_df_copy["defect_type"].map(defect_class_map)
)

train_df_copy.head(10)

Values in 'has_defect': [1 0]

Number of rows having a defect: 7201

Dataset shape: (45572, 5)

Dataset columns:
Index(['image_id', 'defect_type', 'mask_rle', 'width', 'height'], dtype='object')


,image_id,defect_type,mask_rle,width,height,has_defect,defect_class
0,4S22I7SA34,defect_a,7255 6 7377 12 7505 12 7633 12 7762 11 7890 11...,800,128,1,1
1,4S22I7SA34,defect_b,NaN,800,128,0,0
2,4S22I7SA34,defect_c,NaN,800,128,0,0
3,4S22I7SA34,defect_d,NaN,800,128,0,0
4,DZ899111OE,defect_a,NaN,800,128,0,0
5,DZ899111OE,defect_b,NaN,800,128,0,0
6,DZ899111OE,defect_c,NaN,800,128,0,0
7,DZ899111OE,defect_d,NaN,800,128,0,0
8,GVDJ3PQ5QT,defect_a,NaN,800,128,0,0
9,GVDJ3PQ5QT,defect_b,NaN,800,128,0,0


In [7]:
image_defect_sum = train_df_copy.groupby("image_id")["defect_class"].sum()
print(f"No of images having no defects clean images :", (image_defect_sum == 0).sum())
print(f"No of Images having atleast one defect:",(image_defect_sum > 0).sum() )

No of images having no defects clean images : 5516
No of Images having atleast one defect: 5877


In [8]:
defect_counts = (
    train_df_copy[train_df_copy['has_defect'] == 1]
    .groupby('defect_type')['image_id']
    .nunique()
    .sort_values(ascending=False)
)

print(defect_counts)

defect_type
defect_c    4100
defect_d    1707
defect_a    1081
defect_b     313
Name: image_id, dtype: int64


In [9]:
defect_rows = train_df_copy[train_df_copy["defect_class"] > 0]

image_defect_combinations = (defect_rows.groupby("image_id")["defect_type"].apply(lambda x: tuple(sorted(x))))

combination_counts = image_defect_combinations.value_counts()

multi_defect_combinations = combination_counts[combination_counts.index.map(len) >= 2]
multi_defect_combinations

defect_type
(defect_c, defect_d)              598
(defect_a, defect_c)              215
(defect_a, defect_d)              136
(defect_a, defect_c, defect_d)     87
(defect_b, defect_c)               67
(defect_b, defect_d)               33
(defect_a, defect_b)               23
(defect_b, defect_c, defect_d)     20
(defect_a, defect_b, defect_d)     11
(defect_a, defect_b, defect_c)      8
Name: count, dtype: int64

In [10]:
multi_defect_combinations.sum()

np.int64(1198)

## Rle_decode - Rle_encode

In [11]:
def rle_decode(rle_string):
    shape =(128, 800)
    height, width = shape

    mask = np.zeros(height*width, dtype = np.uint8)

    if rle_string:
        rle_list_string = rle_string.strip().split()
        rle_list = [int(x) for x in rle_list_string]
        starts = rle_list[::2] 
        lengths = rle_list[1::2] 

    else:
        return mask.reshape(shape)

    for start, length in zip(starts, lengths):
        mask[start -1 : start + length - 1] = 1

    return mask.reshape(shape, order='F')


def rle_encode(mask):
    pixels = mask.flatten(order="F")
    padded = np.concatenate([[0], pixels, [0]])

    changes = np.where(padded[1:] != padded[:-1])[0]

    changes[1::2] -= changes[::2]
    changes[::2] += 1

    return " ".join(map(str, changes))

# Dataset

In [13]:
from torch.utils.data import Dataset, DataLoader
class SurfaceAnomalyDataset(Dataset):

    def __init__(self, df, img_dir):

        df = df.copy()
        df["mask_rle"] = df["mask_rle"].fillna("")
        self.img_dir = img_dir
        self.ids = df["image_id"].unique().tolist()

        self.rles = {
            image_id: (
                group.set_index("defect_type")["mask_rle"]
                .reindex(CLASSES)
                .fillna("")
                .tolist()
            )
            for image_id, group in df.groupby("image_id")
        }

    def __len__(self):
        return len(self.ids)
        

        


    def __getitem__(self, idx):

        img_id = self.ids[idx]
        img_path = os.path.join(self.img_dir, f"{img_id}.jpg")

        img = cv2.imread(img_path)

        if img is None:
            raise FileNotFoundError(img_path)
        
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = torch.from_numpy(
            np.ascontiguousarray(img.transpose(2, 0, 1))
        ).float() / 255.0
        masks = np.stack([
            rle_decode(rle)
            for rle in self.rles[img_id]
        ])

        masks = torch.from_numpy(masks).float()

        return img, masks

In [15]:
ds = SurfaceAnomalyDataset(train_df, TRAIN_DIR)

print("Dataset length:", len(ds))

Dataset length: 11393


# Tran val Split 

In [18]:
# train_df_copy.head()
image_labels = (
    train_df_copy.pivot_table(
        index="image_id",
        columns="defect_type",
        values="has_defect",
        aggfunc="max",
        fill_value=0
    )
    .reindex(columns=CLASSES, fill_value=0)
    .astype(int)
)

display(image_labels.head())

defect_type,defect_a,defect_b,defect_c,defect_d
image_id,,,,
000HAM628W,0,0,1,0
005VFS2MW4,0,0,0,1
0070O9IJVX,0,0,0,1
00761I10TU,0,0,1,0
009DUJJXC1,0,0,0,0


In [22]:
# weights = np.array([1, 2, 4, 8])

# image_labels["combo"] = (
#     image_labels[CLASSES].values @ weights
# )
# image_labels.head()

In [23]:
from sklearn.model_selection import StratifiedKFold

def stratified_split(image_labels, n_splits, fold, seed):

    weights = np.array([1, 2, 4, 8])

    
    combo = image_labels[CLASSES].values @ weights

    
    counts = pd.Series(combo).value_counts()
    rare = counts[counts < n_splits].index
    stratified_labels = pd.Series(combo).replace(rare, -1).values

    
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed
    )

    X = image_labels.index.to_numpy()

    for current_fold, (train_idx, val_idx) in enumerate(
        skf.split(X, stratified_labels)
    ):
        if current_fold == fold:
            train_ids = X[train_idx]
            val_ids = X[val_idx]
            return train_ids, val_ids

In [24]:
FOLD = 0
SEED = 42

train_ids, val_ids = stratified_split(
    image_labels,
    n_splits=5,
    fold=FOLD,
    seed=SEED
)

print("Training images:", len(train_ids))
print("Validation images:", len(val_ids))

# Create DataFrames
train_split_df = train_df[
    train_df["image_id"].isin(train_ids)
].copy()

val_split_df = train_df[
    train_df["image_id"].isin(val_ids)
].copy()

print("Train rows:", len(train_split_df))
print("Validation rows:", len(val_split_df))

# Check the split
print("Image overlap:", len(set(train_ids) & set(val_ids)))

for defect in CLASSES:
    train_rate = image_labels.loc[train_ids, defect].mean() * 100
    val_rate = image_labels.loc[val_ids, defect].mean() * 100

    print(
        f"{defect}: "
        f"Train = {train_rate:.2f}% | "
        f"Validation = {val_rate:.2f}%"
    )

Training images: 9114
Validation images: 2279
Train rows: 36456
Validation rows: 9116
Image overlap: 0
defect_a: Train = 9.49% | Validation = 9.48%
defect_b: Train = 2.75% | Validation = 2.72%
defect_c: Train = 35.98% | Validation = 36.02%
defect_d: Train = 14.99% | Validation = 14.96%


In [25]:
train_dataset = SurfaceAnomalyDataset(
    train_split_df,
    TRAIN_DIR
)

val_dataset = SurfaceAnomalyDataset(
    val_split_df,
    TRAIN_DIR
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

Train dataset: 9114
Validation dataset: 2279


# Dataloaders

In [27]:
from torch.utils.data import DataLoader
BATCH_SIZE = 16
NUM_WORKERS = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

In [29]:
import cv2
images, masks = next(iter(train_loader))

print("Images shape:", images.shape)
print("Masks shape:", masks.shape)

print("Image min/max:", images.min().item(), images.max().item())
print("Mask min/max:", masks.min().item(), masks.max().item())

Images shape: torch.Size([16, 3, 128, 800])
Masks shape: torch.Size([16, 4, 128, 800])
Image min/max: 0.0 1.0
Mask min/max: 0.0 1.0


In [30]:
val_images, val_masks = next(iter(val_loader))

print("Validation images:", val_images.shape)
print("Validation masks:", val_masks.shape)

Validation images: torch.Size([16, 3, 128, 800])
Validation masks: torch.Size([16, 4, 128, 800])


# BASELINE U-NET

In [31]:
def DoubleConv(in_channels, out_channels):
    return nn.sequential(
        nn.Conv2d(in_channels, out_channels, kernel=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.Relu(inplace=true),
        
        nn.Conv2d(in_channels, out_channels, kernel=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.Relu(inplace=true),
        
    )

# U-Net architecture

In [ ]:
class UNet(nn.Module):
    def __init__(self, )

In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/input/competitions/industrial-surface-defect-detection/public/sample_submission.csv")
df.to_csv("submission.csv", index=False)